In [5]:
value_counts().to_dict()
            print(f"  {trial_type.upper()}:")
            for soa, count in sorted(soa_counts.items()):
                percent = (count / len(subset)) * 100
                print(f"    {soa} ms: {count} trials ({percent:.1f}%)")
        
        # 4. Jitter distribution
        print("\nJitter distribution (excluding practice):")
        jitter_counts = exp_df['jitter_ms'].value_counts().to_dict()
        for jitter, count in sorted(jitter_counts.items()):
            percent = (count / experimental_count) * 100
            print(f"  {jitter} ms: {count} trials ({percent:.1f}%)")
        
        # 5. Breath phase checks for experimental trials
        print("\nBreath phase distribution (excluding practice):")
        if 'breathphase' in exp_df.columns:
            phase_counts = exp_df['breathphase'].value_counts().to_dict()
            for phase, count in sorted(phase_counts.items()):
                percent = (count / experimental_count) * 100
                print(f"  {phase}: {count} trials ({percent:.1f}%)")
        
        # 6. Cross-tabulation of trial types by breath phase
        print("\nTrial types by breath phase (COUNTERBALANCE CHECK):")
        if 'breathphase' in exp_df.columns:
            trial_by_phase = pd.crosstab(
                exp_df['trial_type'], 
                exp_df['breathphase']
            )
            print(trial_by_phase)
            
            # Calculate chi-square statistic for independence (excluding mandatory assignments)
            try:
                # Create subset without inhalation/exhalation (which are deterministically assigned)
                flex_df = exp_df[exp_df['trial_type'].isin(['baseline', 'catch'])]
                if len(flex_df) > 0:
                    flex_table = pd.crosstab(flex_df['trial_type'], flex_df['breathphase'])
                    chi2 = 0
                    row_sums = flex_table.sum(axis=1)
                    col_sums = flex_table.sum(axis=0) 
                    total = flex_table.sum().sum()
                    
                    for i in flex_table.index:
                        for j in flex_table.columns:
                            observed = flex_table.loc[i, j]
                            expected = row_sums[i] * col_sums[j] / total
                            if expected > 0:
                                chi2 += ((observed - expected) ** 2) / expected
                    
                    print(f"\nChi-square for baseline/catch distribution: {chi2:.2f}")
                    print(f"Lower values indicate better balance between breathphases.")
            except Exception as e:
                print(f"Error calculating chi-square: {e}")
        
        # 7. Consecutive trials analysis (excluding practice)
        trial_sequence = exp_df['trial_type'].tolist()
        consecutive_counts = {}
        
        if trial_sequence:
            current_type = trial_sequence[0]
            current_count = 1
            for t in trial_sequence[1:]:
                if t == current_type:
                    current_count += 1
                else:
                    key = f"{current_type}_{current_count}"
                    consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
                    current_type = t
                    current_count = 1
            key = f"{current_type}_{current_count}"
            consecutive_counts[key] = consecutive_counts.get(key, 0) + 1
            
            print("\nRuns of consecutive trial types (excluding practice):")
            for key, count in sorted(consecutive_counts.items()):
                print(f"  {key}: {count}")
    
    # ------------------------------------------------------------------
    # Process a single participant: generate design, assign slots, finalize, and save CSV.
    # ------------------------------------------------------------------
    def process_participant(self, participant_id):
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            design_df = self.generate_counterbalanced_design(participant_id)
            design_df = self.assign_breath_holds(design_df, participant_id)
            design_df = self.finalize_design_csv(design_df)
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    # ------------------------------------------------------------------
    # Run design generation for multiple participants.
    # ------------------------------------------------------------------
    def run(self, participant_ids=None, num_participants=None):
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator with WAV Marker Integration ===")
            print(f"Marker count: {len(self.wav_markers)}")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Practice trials: {self.config['practice_trials']}")
            print(f"Participants: {participant_ids}")
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {'success': success, 'csv_path': path}
        
        # Summary of all participants
        if self.config['debug_mode'] and len(results) > 1:
            print("\n=== CROSS-PARTICIPANT VALIDATION ===")
            all_success = all(info['success'] for info in results.values())
            print(f"All participants generated successfully: {all_success}")
        
        return results

# ----------------------------------------------------------------------
# Integrated workflow function
# ----------------------------------------------------------------------
def run_integrated_workflow(config):
    """
    Run the entire integrated workflow:
    1. Calculate required repetitions
    2. Create audio files with markers
    3. Generate participant designs
    
    Args:
        config: Configuration dictionary
        
    Returns:
        Dictionary with results
    """
    print("\n===== RUNNING INTEGRATED PPS DESIGN WORKFLOW =====")
    
    # Step 1: Initialize generator to calculate trial counts
    temp_generator = PPSDesignGenerator(config)
    
    # Step 2: Calculate repetitions needed
    repetitions_needed = temp_generator.calculate_required_repetitions()
    config['audio']['repetition_count'] = repetitions_needed
    
    print(f"\n===== AUDIO CREATION PHASE =====")
    print(f"Calculated {repetitions_needed} repetition segments needed")
    
    # Step 3: Create concatenated audio with markers
    concatenated_wav, marker_csv, extracted_markers = create_concatenated_audio_with_markers(config)
    
    print(f"\n===== DESIGN GENERATION PHASE =====")
    
    # Step 4: Initialize final generator with extracted markers
    generator = PPSDesignGenerator(config, extracted_markers)
    
    # Step 5: Generate participant designs
    participant_ids = list(range(config['num_participants']))
    results = generator.run(participant_ids)
    
    # Print summary
    print("\n===== WORKFLOW SUMMARY =====")
    print(f"Created audio files:")
    print(f"  - Concatenated WAV: {concatenated_wav}")
    print(f"  - Marker CSV: {marker_csv}")
    print(f"  - Markers extracted: {len(extracted_markers)}")
    
    success_count = sum(1 for info in results.values() if info['success'])
    print(f"Participant designs generated: {success_count}/{len(results)}")
    
    return {
        'audio_files': {
            'concatenated_wav': concatenated_wav,
            'marker_csv': marker_csv,
            'marker_count': len(extracted_markers)
        },
        'participant_designs': results
    }

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the integrated workflow."""
    # Print configuration summary
    print("\n===== INTEGRATED PPS EXPERIMENT DESIGN GENERATOR =====")
    print(f"Audio files:")
    print(f"  - Intro: {CONFIG['paths']['intro_audio']}")
    print(f"  - Repetition segment: {CONFIG['paths']['repetition_audio']}")
    print(f"  - Intro CSV: {CONFIG['paths']['intro_csv']}")
    print(f"  - Repetition CSV: {CONFIG['paths']['repetition_csv']}")
    print(f"Output directories:")
    print(f"  - Audio: {CONFIG['paths']['experiment_audio_dir']}")
    print(f"  - Design CSVs: {CONFIG['paths']['experiment_log_dir']}")
    print(f"Configuration:")
    print(f"  - Repetitions (trials per SOA per condition): {CONFIG['repetitions']}")
    print(f"  - Practice trials at beginning: {CONFIG['practice_trials']}")
    print(f"  - SOA Conditions: {CONFIG['soa_conditions_ms']}")
    print(f"  - Looming Stimuli: {LOOMING_STIMULI}")
    print(f"  - Number of Participants: {CONFIG['num_participants']}")
    
    # Run the integrated workflow
    results = run_integrated_workflow(CONFIG)
    
    # Show final summary
    print("\n===== COMPLETE =====")
    print(f"Audio file created: {results['audio_files']['concatenated_wav']}")
    print(f"Marker CSV created: {results['audio_files']['marker_csv']}")
    print(f"Participant designs created: {sum(1 for info in results['participant_designs'].values() if info['success'])}")
    print("\nProcess complete!")

if __name__ == "__main__":
    main()


===== INTEGRATED PPS EXPERIMENT DESIGN GENERATOR =====
Audio files:
  - Intro: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_intro.wav
  - Repetition segment: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetitionsegment.wav
  - Intro CSV: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv
  - Repetition CSV: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetition_timestamps.csv
Output directories:
  - Audio: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio
  - Design CSVs: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
Configuration:
  - Repetitions (trials per SOA per condition): 12
  - Practice trials at beginning: 8
  - SOA Conditions: [190, 400, 700, 1000, 1500]
  - Looming Stimuli: ['blue', 'brown', 'pink', 'white'